In [ ]:
import graphite


In [ ]:
dataset = graphite.Dataset(source='washington',
                           level='line',
                           training_ratio=None,
                           validation_ratio=None,
                           test_ratio=None,
                           lazy_mode=False,
                           seed=42)
print(dataset)


In [ ]:
augmentor = graphite.Augmentor(  # elastic_transform=[0.99, 43, 0.75],
    erosion=[0.99, 5, 1],
    dilation=[0.99, 2, 1],
    #    mixup=[0.99, 0.3, 1],
    #    perspective_transform=[0.99, 0.4],
    #    salt_and_pepper=[0.99, 0.3],
    #    gaussian_blur=[0.99, 11, 1],
    #    shearing=[0.99, 30],
    scaling=[0.99, 0.05],
    rotation=[0.99, 0.5],
    translation=[0.99, 0.05, 0.05],
    seed=42)
print(augmentor)


In [ ]:
model = graphite.Model(network='flor', experiment_name='Tutorial', seed=42)

model.compile(tokenizer=dataset.tokenizer, learning_rate=1e-3)

print(model)


In [ ]:
train_data, train_steps = dataset.get_generator(dataset.training, batch_size=16, augmentor=augmentor)
valid_data, valid_steps = dataset.get_generator(dataset.validation, batch_size=16, augmentor=None)

model.fit(training_data=train_data,
          training_steps=train_steps,
          validation_data=valid_data,
          validation_steps=valid_steps,
          plateau_factor=0.5,
          plateau_cooldown=10,
          plateau_patience=20,
          patience=60,
          epochs=1,
          verbose=1)


In [ ]:
test_data, test_steps = dataset.get_generator(dataset.test, batch_size=16, augmentor=None)

predictions, probabilities = model.predict(test_data=test_data,
                                           test_steps=test_steps,
                                           top_paths=1,
                                           beam_width=30,
                                           ctc_decode=True,
                                           token_decode=True,
                                           verbose=1)

print(predictions)


In [ ]:

metrics, _ = model.evaluate(dataset.test,
                            predictions=predictions,
                            share_top_paths=True,
                            prediction_samples=10,
                            origin='vanilla')

print(metrics)


In [ ]:
spelling = graphite.Spelling(spell_checker='openai', env_key='OPENAI_API_KEY')
enhanced_predictions = spelling.enhance(predictions)

enhanced_predictions


In [ ]:
enhanced_metrics, _ = model.evaluate(dataset.test,
                                     predictions=enhanced_predictions,
                                     share_top_paths=True,
                                     prediction_samples=10,
                                     origin='openai')

print(enhanced_metrics)


In [ ]:
model.save_context(dataset=dataset,
                   augmentor=augmentor,
                   metrics=metrics,
                   enhanced_metrics=enhanced_metrics)


In [ ]:
# import os
# import json
# import mlflow
# import datetime

# # import logging
# # logging.getLogger("mlflow").setLevel(logging.DEBUG)


# # experiment = mlflow.set_experiment(experiment_name='Tutorial')
# # run_name = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
# # run = None

# model.run

# # with mlflow.start_run(run_id=model.run.info.run_id, experiment_id=model.experiment.experiment_id):
# print(model.run.info.artifact_uri)

# artifacts_uri = model.run.info.artifact_uri.replace('file://', '')
# os.makedirs(artifacts_uri, exist_ok=True)

# # Metrics
# dict_metrics = {}

# for i, metric in enumerate(metrics):
#     dict_metrics[f"top_path_{i+1}_cer"] = metric[0]
#     dict_metrics[f"top_path_{i+1}_wer"] = metric[1]
#     dict_metrics[f"top_path_{i+1}_ler"] = metric[2]
#     dict_metrics[f"top_path_{i+1}_ser"] = metric[3]

# for i, metric in enumerate(enhanced_metrics):
#     dict_metrics[f"enhanced_top_path_{i+1}_cer"] = metric[0]
#     dict_metrics[f"enhanced_top_path_{i+1}_wer"] = metric[1]
#     dict_metrics[f"enhanced_top_path_{i+1}_ler"] = metric[2]
#     dict_metrics[f"enhanced_top_path_{i+1}_ser"] = metric[3]

# mlflow.log_metrics(metrics=dict_metrics)

# # Parameters
# metadata_uri = os.path.join(artifacts_uri, 'metadata.json')
# os.makedirs(os.path.dirname(metadata_uri), exist_ok=True)

# dict_params = {
#     **dataset.__repr__(),
#     **dataset.tokenizer.__repr__(),
#     **augmentor.__repr__(),
#     **model.__repr__(),
#     **model.training_logger.__repr__(),
#     **model.test_logger.__repr__(),
# }

# mlflow.log_params(dict_params)

# # Logs
# filelogs = [
#     (dataset, 'dataset.log', 'w'),
#     (dataset.tokenizer, 'tokenizer.log', 'w'),
#     (augmentor, 'augmentor.log', 'w'),
#     (model, 'model.log', 'w'),
#     (model.training_logger, 'training.log', 'w'),
#     (model.test_logger, 'test.log', 'a'),
#     (model.samples_logger, 'samples.log', 'w'),
# ]

# for log, filename, open_mode in filelogs:
#     log_uri = os.path.join(artifacts_uri, 'logs', filename)
#     os.makedirs(os.path.dirname(log_uri), exist_ok=True)

#     with open(log_uri, open_mode) as f:
#         f.write(f"{str(log)}".strip())

#     mlflow.log_artifact(log_uri, 'logs')

# # Tokenizer
# import pickle
# tokenizer_uri = os.path.join(artifacts_uri, 'tokenizer.pkl')
# os.makedirs(os.path.dirname(tokenizer_uri), exist_ok=True)

# with open(tokenizer_uri, 'wb') as f:
#     pickle.dump(model.tokenizer, f)

# mlflow.log_artifact(tokenizer_uri)

# # Model
# model_uri = os.path.join(artifacts_uri, 'model.keras')
# os.makedirs(os.path.dirname(model_uri), exist_ok=True)

# model.model.save(model_uri)
# mlflow.log_artifact(model_uri)

# # Tag
# mlflow.set_tags({'mlflow.model': model.network})


In [ ]:
# runs_df = mlflow.search_runs(experiment_ids=[model.experiment.experiment_id],
#                              filter_string=f"tags.mlflow.model='{model.network}'",
#                              order_by=['tags.mlflow.runName ASC'])

# run = mlflow.get_run(runs_df.iloc[0]['run_id'])
# # runs_df

# os.path.join(run.info.artifact_uri, 'model.keras')

# # # predict
# # model = mlflow.pyfunc.load_model(model_uri=best_run.info.artifact_uri)

# # model.predict()
# # best_run.info.artifact_uri

# # df_pred = model.predict(df)
